# 4. Model Deployment
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Loading the best model (Tuned Random Forest)
- Local inference testing
- SageMaker Model Registry
- SageMaker Endpoint deployment
- Batch Transform job
- Endpoint invocation and testing

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

## 4.1 Load Best Model & Test Data

In [ ]:
best_model = joblib.load('../models/random_forest_best.joblib')
test_data = pd.read_csv('../data/processed/test.csv')

X_test = test_data.drop(columns=['performance'])
y_test = test_data['performance']

print(f'Model loaded: {type(best_model).__name__}')
print(f'Model parameters: {best_model.get_params()}')
print(f'Test set shape: {X_test.shape}')

## 4.2 Local Inference Test

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

predictions = best_model.predict(X_test)
probabilities = best_model.predict_proba(X_test)

print('=== Local Inference Results ===')
print(f'Test Accuracy: {accuracy_score(y_test, predictions):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, predictions, target_names=['Low', 'High']))

print('\nSample Predictions (first 10):')
sample_results = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': predictions[:10],
    'Prob_Low': probabilities[:10, 0].round(3),
    'Prob_High': probabilities[:10, 1].round(3)
})
print(sample_results.to_string(index=False))

## 4.3 Prepare Model Artifact for SageMaker

In [ ]:
import tarfile

os.makedirs('../models/sagemaker', exist_ok=True)

# SageMaker expects model.joblib inside a model.tar.gz
model_path = '../models/random_forest_best.joblib'
tar_path = '../models/sagemaker/model.tar.gz'

with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(model_path, arcname='model.joblib')

print(f'Model artifact created: {tar_path}')
print(f'Size: {os.path.getsize(tar_path) / 1024:.1f} KB')

## 4.4 SageMaker Model Deployment
Uncomment when AWS access is available.

In [ ]:
# import sagemaker
# import boto3
# from sagemaker.sklearn import SKLearnModel
# from sagemaker import get_execution_role
#
# session = sagemaker.Session()
# bucket = session.default_bucket()
# role = get_execution_role()
# prefix = 'student-performance'
#
# # Upload model artifact to S3
# model_artifact = session.upload_data(
#     path='../models/sagemaker/model.tar.gz',
#     bucket=bucket,
#     key_prefix=f'{prefix}/model'
# )
# print(f'Model artifact uploaded to: {model_artifact}')

In [ ]:
# # Create SageMaker SKLearn Model
# sklearn_model = SKLearnModel(
#     model_data=model_artifact,
#     role=role,
#     framework_version='1.2-1',
#     entry_point='../src/inference.py',
#     sagemaker_session=session
# )
#
# print('SKLearn Model created')

### 4.4.1 Register Model in SageMaker Model Registry

In [ ]:
# from sagemaker.model_metrics import ModelMetrics, MetricsSource
#
# model_package_group_name = 'StudentPerformanceModelGroup'
#
# # Register model
# model_package = sklearn_model.register(
#     model_package_group_name=model_package_group_name,
#     content_types=['text/csv'],
#     response_types=['text/csv'],
#     inference_instances=['ml.t2.medium', 'ml.m5.large'],
#     transform_instances=['ml.m5.large'],
#     approval_status='Approved',
#     description='Student Performance Prediction - Tuned Random Forest'
# )
#
# print(f'Model registered: {model_package.model_package_arn}')

### 4.4.2 Deploy to Real-Time Endpoint

In [ ]:
# # Deploy endpoint
# predictor = sklearn_model.deploy(
#     initial_instance_count=1,
#     instance_type='ml.t2.medium',
#     endpoint_name='student-performance-endpoint'
# )
#
# print(f'Endpoint deployed: {predictor.endpoint_name}')

### 4.4.3 Test Endpoint Invocation

In [ ]:
# from sagemaker.serializers import CSVSerializer
# from sagemaker.deserializers import JSONDeserializer
#
# predictor.serializer = CSVSerializer()
# predictor.deserializer = JSONDeserializer()
#
# # Test with sample data
# sample = X_test.iloc[:5].values
# result = predictor.predict(sample)
#
# print('=== Endpoint Invocation Results ===')
# print(f'Input shape: {sample.shape}')
# print(f'Predictions: {result}')
# print(f'Actual:      {y_test.values[:5]}')

### 4.4.4 Batch Transform Job

In [ ]:
# # Upload test data to S3 for batch transform
# test_no_target = X_test.copy()
# test_no_target.to_csv('../data/processed/batch_input.csv', index=False, header=False)
#
# batch_input = session.upload_data(
#     path='../data/processed/batch_input.csv',
#     bucket=bucket,
#     key_prefix=f'{prefix}/batch-input'
# )
#
# # Create Batch Transform job
# transformer = sklearn_model.transformer(
#     instance_count=1,
#     instance_type='ml.m5.large',
#     output_path=f's3://{bucket}/{prefix}/batch-output'
# )
#
# transformer.transform(
#     data=batch_input,
#     content_type='text/csv',
#     split_type='Line'
# )
#
# transformer.wait()
# print(f'Batch Transform complete. Output: s3://{bucket}/{prefix}/batch-output')

### 4.4.5 Cleanup — Delete Endpoint
**Important:** Always delete the endpoint after use to avoid ongoing charges.

In [ ]:
# # DELETE ENDPOINT after demo recording!
# predictor.delete_endpoint()
# print('Endpoint deleted successfully.')

## Summary
- Loaded and verified best model (Tuned Random Forest)
- Local inference validated on test set
- Model artifact packaged as model.tar.gz for SageMaker
- SageMaker deployment code ready: Model Registry, Real-time Endpoint, Batch Transform
- Endpoint cleanup code included to prevent unnecessary charges